In [3]:
import pandas as pd
import re
import numpy as np
from bs4 import BeautifulSoup
import time
import requests


# En este notebook vamos a recopilar datos y convertirlos en csv  
# para posteriormente juntarlos y analizarlos

In [ ]:
animes_csv = pd.read_csv('C:/Users/andre\Desktop\Curso_Data_Analytics\Tercera Semana\Proyecto Anime/anime.csv')
animes_csv.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [77]:
url = "https://myanimelist.net/topanime.php"
response = requests.get(url)
response

<Response [200]>

In [17]:
soup = BeautifulSoup(response.content, "html.parser")
soup


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

<html class="appearance-none" lang="en">
<head>
<link crossorigin="anonymous" href="//www.googletagmanager.com/" rel="preconnect"/>
<link crossorigin="anonymous" href="https://cdn.myanimelist.net" rel="preconnect"/>
<title>
  Top Anime - MyAnimeList.net
</title>
<meta content="Browse the highest-ranked anime on MyAnimeList, the internet's largest anime database. Find the top TV series, movies, and OVAs right here!" name="description"/>
<meta content="anime, myanimelist, anime news, manga" name="keywords"/>
<link href="https://myanimelist.net/topanime.php?limit=50" rel="next"/>
<meta content="en_US" property="og:locale"/><meta content="360769957454434" property="fb:app_id"/><meta content="MyAnimeList.net" property="og:site_name"/><meta content="summary" name="twitter:card"/><meta content="@myanimelist" name="twitter:site"/><meta content=" Top Anime - MyAnimeLis

In [ ]:
animes1 = soup.find_all("div", class_= "detail")
len(animes1)

50

In [50]:
animes1[0]

<div class="detail"><div id="area52991">
<div class="hoverinfo" id="info52991" rel="a52991"></div>
</div>
<div class="di-ib clearfix"><h3 class="fl-l fs14 fw-b anime_ranking_h3"><a class="hoverinfo_trigger" href="https://myanimelist.net/anime/52991/Sousou_no_Frieren" id="#area52991" rel="#info52991">Sousou no Frieren</a></h3><div class="icon-watch2"><a class="mal-icon ml8 ga-click" href="https://myanimelist.net/anime/52991/Sousou_no_Frieren/video" title="Watch Episode Video"><i class="malicon malicon-movie-episode"></i></a></div></div><br/><div class="information di-ib mt4">
        TV (28 eps)<br/>
        Sep 2023 - Mar 2024<br/>
        1,238,241 members
      </div> </div>

## Vamos a encontrar el titulo de un anime. Para ello: 

In [37]:
animes1[0].find('h3', class_='anime_ranking_h3')

<h3 class="fl-l fs14 fw-b anime_ranking_h3"><a class="hoverinfo_trigger" href="https://myanimelist.net/anime/52991/Sousou_no_Frieren" id="#area52991" rel="#info52991">Sousou no Frieren</a></h3>

In [39]:
titulo = animes1[0].a.text.strip()
titulo

'Sousou no Frieren'

## Ahora vamos con el año de estreno:

In [ ]:
#Año de estreno:
info_block = soup.find('div', class_='information')

# Obtenemos el texto limpio
info_text = info_block.get_text(separator='\n').strip()

# Dividimos en líneas no vacías
info_lines = [line.strip() for line in info_text.split('\n') if line.strip()]



# Extraer el primer año
year = re.search(r'\b(19|20)\d{2}\b', fecha_linea).group(0) if fecha_linea else None
year

'2023'

## Y ahora con la valoración, que se encuentra en otro objeto:

In [ ]:
#Rating
ratings = soup.find_all("span", class_=re.compile(r"score-label"))

#Pero al parecer hay valoraciones ocultas que nos aparecen como N/A, por lo que necesitamos filtrarlas
ratings_validos = [r.text.strip() for r in ratingsif re.match(r'^\d+(\.\d+)?$', r.text.strip())]
for r in ratings_validos:print(r)
df_ratings = pd.DataFrame(ratings_validos, columns=['rating'])


8.50
8.50
8.50
8.50
8.50
8.49
8.49
8.49
8.48
8.48
8.48
8.48
8.47
8.47
8.47
8.47
8.47
8.47
8.46
8.46
8.46
8.46
8.46
8.46
8.46
8.45
8.45
8.45
8.45
8.45
8.45
8.44
8.44
8.44
8.44
8.44
8.44
8.43
8.43
8.43
8.43
8.42
8.42
8.42
8.42
8.42
8.42
8.42
8.42
8.42


## Todo junto quedaría así:

In [ ]:
# ---- Extraer títulos y años ----
details = soup.find_all('div', class_='detail')

titulos = []
anios = []
for detail in details:
    # ---- Título ----
    titulo_tag = detail.find('h3', class_='anime_ranking_h3')
    titulo = titulo_tag.a.text.strip() if titulo_tag and titulo_tag.a else None

    # ---- Año ----
    info_block = detail.find('div', class_='information')
    anio = None
    if info_block:
        info_text = info_block.get_text(separator='\n').strip()
        match = re.search(r'\b(19|20)\d{2}\b', info_text)
        if match:
            anio = match.group(0)

    titulos.append(titulo)
    anios.append(anio)

# ---- Extraer todos los ratings válidos ----
ratings = soup.find_all("span", class_=re.compile(r"score-label"))
ratings_validos = [r.text.strip()for r in ratings if re.match(r'^\d+(\.\d+)?$', r.text.strip())]

# ---- Emparejar todo ----
animes = []
for titulo, anio, rating in zip(titulos, anios, ratings_validos):
    animes.append({'titulo': titulo,'año': anio,'rating': rating})

# ---- Mostrar resultado ----
for a in animes:
    print(a)


{'titulo': 'Sousou no Frieren', 'año': '2023', 'rating': '9.29'}
{'titulo': 'Fullmetal Alchemist: Brotherhood', 'año': '2009', 'rating': '9.10'}
{'titulo': 'Steins;Gate', 'año': '2011', 'rating': '9.07'}
{'titulo': 'Shingeki no Kyojin Season 3 Part 2', 'año': '2019', 'rating': '9.05'}
{'titulo': 'Gintama: The Final', 'año': '2021', 'rating': '9.05'}
{'titulo': 'Gintama°', 'año': '2015', 'rating': '9.05'}
{'titulo': 'Hunter x Hunter (2011)', 'año': '2011', 'rating': '9.03'}
{'titulo': 'One Piece Fan Letter', 'año': '2024', 'rating': '9.03'}
{'titulo': 'Chainsaw Man Movie: Reze-hen', 'año': '2025', 'rating': '9.02'}
{'titulo': 'Ginga Eiyuu Densetsu', 'año': '1988', 'rating': '9.02'}
{'titulo': "Gintama'", 'año': '2011', 'rating': '9.02'}
{'titulo': "Gintama': Enchousen", 'año': '2012', 'rating': '9.02'}
{'titulo': 'Bleach: Sennen Kessen-hen', 'año': '2022', 'rating': '8.99'}
{'titulo': 'Gintama.', 'año': '2017', 'rating': '8.98'}
{'titulo': 'Kaguya-sama wa Kokurasetai: Ultra Romantic', '

## Por último, conseguir el género y repetir todo el proceso para los 200 primeros animes

In [79]:
for anime in animes1:
    print(anime)

<div class="detail"><div id="area52991">
<div class="hoverinfo" id="info52991" rel="a52991"></div>
</div>
<div class="di-ib clearfix"><h3 class="fl-l fs14 fw-b anime_ranking_h3"><a class="hoverinfo_trigger" href="https://myanimelist.net/anime/52991/Sousou_no_Frieren" id="#area52991" rel="#info52991">Sousou no Frieren</a></h3><div class="icon-watch2"><a class="mal-icon ml8 ga-click" href="https://myanimelist.net/anime/52991/Sousou_no_Frieren/video" title="Watch Episode Video"><i class="malicon malicon-movie-episode"></i></a></div></div><br/><div class="information di-ib mt4">
        TV (28 eps)<br/>
        Sep 2023 - Mar 2024<br/>
        1,238,241 members
      </div> </div>
<div class="detail"><div id="area5114">
<div class="hoverinfo" id="info5114" rel="a5114"></div>
</div>
<div class="di-ib clearfix"><h3 class="fl-l fs14 fw-b anime_ranking_h3"><a class="hoverinfo_trigger" href="https://myanimelist.net/anime/5114/Fullmetal_Alchemist__Brotherhood" id="#area5114" rel="#info5114">Full

In [80]:
animes1[0].find("h3").find("a").get("href")

'https://myanimelist.net/anime/52991/Sousou_no_Frieren'

In [84]:
url2= animes1[0].find("h3").find("a").get("href")
url2

'https://myanimelist.net/anime/52991/Sousou_no_Frieren'

In [ ]:
response2 = requests.get(url2)
soup2 = BeautifulSoup(response2.content, "html.parser")
soup2


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

<html class="appearance-none" lang="en" xmlns:fb="http://www.facebook.com/2008/fbml" xmlns:og="http://ogp.me/ns#">
<head>
<link crossorigin="anonymous" href="//www.googletagmanager.com/" rel="preconnect"/>
<link crossorigin="anonymous" href="https://cdn.myanimelist.net" rel="preconnect"/>
<title>
Sousou no Frieren (Frieren: Beyond Journey's End) - MyAnimeList.net
</title>
<meta content="Looking for information on the anime Sousou no Frieren (Frieren: Beyond Journey's End)? Find out more with MyAnimeList, the world's most active online anime and manga community and database. During their decade-long quest to defeat the Demon King, the members of the hero's party—Himmel himself, the priest Heiter, the dwarf warrior Eisen, and the elven mage Frieren—forge bonds through adventures and battles, creating unforgettable precious memories for most of them. However, the

In [97]:
genre_block = None
for div in soup2.find_all("div", class_="spaceit_pad"):
    label = div.find("span", class_="dark_text")
    if label and "Genres" in label.text:
        genre_block = div
        break
genres = []
if genre_block:
    for a in genre_block.find_all("a"):
        genres.append(a.text.strip())

print(genres)

['Adventure', 'Drama', 'Fantasy']


## Al final juntamos todo y hacemos web scraping 
## en las primeras 4 páginas para obterner 200 animes:

In [ ]:
# ---- Configuración ----
base_url = "https://myanimelist.net/topanime.php"
headers = {"User-Agent": "Mozilla/5.0"}
animes = []

# ---- Recorremos 4 páginas (200 animes) ----
for offset in range(0, 200, 50):  # 0, 50, 100, 150
    url = f"{base_url}?limit={offset}" if offset > 0 else base_url
    print(f"\nScrapeando página: {url}")

    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    details = soup.find_all('div', class_='detail')

    for detail in details:
        # ---- Título y enlace ----
        titulo_tag = detail.find('h3', class_='anime_ranking_h3')
        titulo = titulo_tag.a.text.strip() if titulo_tag and titulo_tag.a else None
        url2 = titulo_tag.a.get('href') if titulo_tag and titulo_tag.a else None

        # ---- Año ----
        info_block = detail.find('div', class_='information')
        anio = None
        if info_block:
            info_text = info_block.get_text(separator='\n').strip()
            match = re.search(r'\b(19|20)\d{2}\b', info_text)
            if match:
                anio = match.group(0)
        # ---- Géneros ----
        genres = []
        if url2:
            try:
                response2 = requests.get(url2, headers=headers)
                soup2 = BeautifulSoup(response2.content, "html.parser")

                genre_block = None
                for div in soup2.find_all("div", class_="spaceit_pad"):
                    label = div.find("span", class_="dark_text")
                    if label and "Genres" in label.text:
                        genre_block = div
                        break
                if genre_block:
                    for a in genre_block.find_all("a"):
                        genres.append(a.text.strip())
                time.sleep(1)  # Pausa entre peticiones a páginas individuales
            except Exception as e:
                print(f"Error al obtener géneros de {url2}: {e}")

        # ---- Guardamos ----
        animes.append({'titulo': titulo,'año': anio,'generos': ', '.join(genres) if genres else None})
    time.sleep(2)  # Pausa entre páginas principales para no saturar el servidor (me estaba dando errores y tiempos de espera muy largos)
    
# ---- Exportar resultados ----
df = pd.DataFrame(animes)
df.to_csv('top_animes_con_generos.csv', index=False, encoding='utf-8-sig')

print(f"\nSe han guardado {len(df)} animes en 'top_animes_con_generos.csv'")




Scrapeando página: https://myanimelist.net/topanime.php

Scrapeando página: https://myanimelist.net/topanime.php?limit=50

Scrapeando página: https://myanimelist.net/topanime.php?limit=100

Scrapeando página: https://myanimelist.net/topanime.php?limit=150

Se han guardado 200 animes en 'top_animes_con_generos.csv'


## Como los ratings me dan problemas, los voy  a sacar aparte y luego los junto

In [12]:
# Configuración
base_url = "https://myanimelist.net/topanime.php"
headers = {"User-Agent": "Mozilla/5.0"}

ratings_totales = []

# Recorrer las 4 primeras páginas (50 animes por página)
for offset in range(0, 200, 50):  # 0, 50, 100, 150
    url = f"{base_url}?limit={offset}" if offset > 0 else base_url
    print(f"Scrapeando página: {url}")
    
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")
    
    # Encontrar todos los ratings
    rating_tags = soup.find_all("span", class_=re.compile(r"score-label"))
    
    # Filtrar solo los valores numéricos
    ratings_validos = [
        r.text.strip() for r in rating_tags
        if re.match(r'^\d+(\.\d+)?$', r.text.strip())
    ]
    
    # Añadir al listado total
    ratings_totales.extend(ratings_validos)
    
    time.sleep(2)  # Pausa para no saturar el servidor

# Crear DataFrame
df_ratings = pd.DataFrame(ratings_totales, columns=['rating'])
print(df_ratings)
print(f"\nTotal de ratings recolectados: {len(df_ratings)}")

Scrapeando página: https://myanimelist.net/topanime.php
Scrapeando página: https://myanimelist.net/topanime.php?limit=50
Scrapeando página: https://myanimelist.net/topanime.php?limit=100
Scrapeando página: https://myanimelist.net/topanime.php?limit=150
    rating
0     9.29
1     9.10
2     9.07
3     9.05
4     9.05
..     ...
195   8.42
196   8.42
197   8.42
198   8.42
199   8.42

[200 rows x 1 columns]

Total de ratings recolectados: 200


In [13]:
df_ratings.head()

,rating
0,9.29
1,9.10
2,9.07
3,9.05
4,9.05


In [14]:
df_animes = pd.read_csv("C:/Users/andre\Desktop\Curso_Data_Analytics\Tercera Semana\Proyecto Anime/top_animes_con_generos.csv")

In [15]:
df_animes.head()

,titulo,año,generos
0,Sousou no Frieren,2023,"Adventure, Drama, Fantasy"
1,Fullmetal Alchemist: Brotherhood,2009,"Action, Adventure, Drama, Fantasy"
2,Steins;Gate,2011,"Drama, Sci-Fi, Suspense"
3,Shingeki no Kyojin Season 3 Part 2,2019,"Action, Drama, Suspense"
4,Gintama: The Final,2021,"Action, Comedy, Drama, Sci-Fi"


In [17]:
df_animes['rating'] = df_ratings['rating']
df_animes

,titulo,año,generos,rating
0,Sousou no Frieren,2023,"Adventure, Drama, Fantasy",9.29
1,Fullmetal Alchemist: Brotherhood,2009,"Action, Adventure, Drama, Fantasy",9.10
2,Steins;Gate,2011,"Drama, Sci-Fi, Suspense",9.07
3,Shingeki no Kyojin Season 3 Part 2,2019,"Action, Drama, Suspense",9.05
4,Gintama: The Final,2021,"Action, Comedy, Drama, Sci-Fi",9.05
...,...,...,...,...
195,Wu Liuqi: Xuanwu Guo Pian,2021,"Action, Adventure, Comedy, Drama, Mystery",8.42
196,Kimetsu no Yaiba,2019,"Action, Award Winning, Supernatural",8.42
197,Kono Oto Tomare! Part 2,2019,NaN,8.42
198,Koukaku Kidoutai: Stand Alone Complex,2002,"Award Winning, Mystery, Sci-Fi",8.42


In [ ]:
df_animes.to_csv('animes y género.csv', index=False)